In [42]:
# Cargo los datos de pedidos
import pandas as pd
from scipy.stats import chi2_contingency

df = pd.read_csv("../data/processed/pedidos_unidos.csv")

print(df.head())

cols_fecha = ["order_purchase_timestamp",	"order_approved_at",	"order_delivered_carrier_date",	"order_delivered_customer_date",	"order_estimated_delivery_date"]


                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [12]:
# Convierto las columnas de fecha a datetime (de str a datetime) para poder leer mejor.

for pedidos in cols_fecha:
    df[pedidos] = pd.to_datetime(df[pedidos])

df.info()


    

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   customer_state                 99441 non-null  str           
 9   customer_city                  99441 non-null  str           
 10  n_items                        98666 non-null  float64       
 11  precio_total              

In [ ]:
# Paso a filtrar por pedidos entregados. 

print(df.value_counts("order_status")) # Me fijo cuantos entregados hay.
mask = df["order_status"] == "delivered"
mask.sum()
df_ent = df[mask]
print(df_ent.shape)
print(df_ent.isna().sum())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
(96478, 17)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date        2
order_delivered_customer_date       8
order_estimated_delivery_date       0
customer_state                      0
customer_city                       0
n_items                             0
precio_total                        0
flete_total                         0
n_vendedores                        0
peso_total_g                        0
categoria                        1332
estado_vendedor                     0
dtype: int64


In [25]:
# Paso a eliminar las 8 filas con nulos en order_delivered_customer_date

df_ent = df_ent.dropna(subset=["order_delivered_customer_date"]).copy()
print(df_ent)



                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
99436  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
99437  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
99438  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
99439  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
99440  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_status order_purchase_timestamp   order_approved_at  \
0       

In [ ]:
# Creo una columna nueva llamada "tarde" para comparar los pedidos que llegaron tarde. 
df_ent["tarde"] = (df_ent["order_delivered_customer_date"] > df_ent["order_estimated_delivery_date"]).astype(int)
df_ent["tarde"].value_counts(normalize = True) # casi 91% de pedidos a tiempo, lo cual me daria un modelo que siempre diria "a tiempo". Target desbalanceado. 


tarde
0    0.918876
1    0.081124
Name: proportion, dtype: float64

In [ ]:
# Aca comparo SOLO EL DIA de entrega, SIN LA HORA.
df_ent["tarde_dia"] = (df_ent["order_delivered_customer_date"].dt.normalize() > df_ent["order_estimated_delivery_date"]).astype(int) # normalize deja la hora real en 00:00:00
print(df_ent["tarde"].mean() * 100)
print(df_ent["tarde_dia"].mean() * 100)

# Uso tarde_dia como target ya que, tecnicamente, se cumple con el criterio de que llego el dia X. 
# Con la definición con hora, tendrías aprox. 1.290 falsas alarmas. 
# Un modelo entrenado así avisaría de retrasos que no son tales, y el equipo de logística perdería confianza en él.

8.112366538820359
6.7730900798175595


In [ ]:
# Primer EDA

df_ent.groupby("customer_state")["tarde_dia"].mean().sort_values(ascending = False) # Porcentaje por estado en Brasil


customer_state
AL    0.214106
MA    0.174338
SE    0.152239
PI    0.138655
CE    0.137608
RR    0.121951
BA    0.121622
RJ    0.121053
PA    0.112051
ES    0.107268
PB    0.104449
TO    0.098540
MS    0.097004
PE    0.096045
RN    0.092827
SC    0.082064
GO    0.065406
RS    0.060816
MT    0.059819
DF    0.056731
MG    0.045711
SP    0.044945
PR    0.040423
AC    0.037500
AP    0.029851
RO    0.028807
AM    0.027586
Name: tarde_dia, dtype: float64

In [ ]:
# Pedidos por estados para saber que porcentajes de retraso son fiables (estados con pocos pedidos no se interpretan)
df_ent.groupby("customer_state").size().sort_values(ascending=False)

customer_state
SP    40494
RJ    12350
MG    11354
RS     5344
PR     4923
SC     3546
BA     3256
DF     2080
ES     1995
GO     1957
PE     1593
CE     1279
PA      946
MT      886
MA      717
MS      701
PB      517
PI      476
RN      474
AL      397
SE      335
TO      274
RO      243
AM      145
AC       80
AP       67
RR       41
dtype: int64

In [45]:
# Armar tabla con crosstab y chi2

tabla_contingencia = pd.crosstab(df_ent["customer_state"], df_ent["tarde_dia"])

chi2, p, grados, esperada = chi2_contingency(tabla_contingencia)

print(chi2)
print(p)
print(grados)

print((esperada < 5).sum())

# Test de chi2 entre customer_tate y tarde_dia donde el retraso depende del estado pero la asociacion es modesta por lo que tengo que combinarlo con otras variables.



1785.7927506269548
0.0
26
2
